# Pandas: Split/Apply/Combine
---

## Overview
In this notebook, we will analyze a "relatively" large (~1 million-row) `DataFrame`, consisting of all five-minute NYS Mesonet observations from August 2025. We will make heavy use of Pandas' powerful `groupby` function, which will *split* the dataset into smaller groups; *apply* functions on the groups; and then *combine* the results of these functions into a new `DataFrame`.

## Prerequisites

| Concepts | Importance | Notes |
| --- | --- | --- |
| Pandas| Necessary | |

* **Time to learn**: 20 minutes

## Imports

In [1]:
import pandas as pd
import numpy as np
import metpy.calc as mpcalc
from metpy.units import units
from datetime import datetime

####  Read in NYSM 5-minute data from a past month. Use the default row and column index names, but convert the date/time data from `String` to `datetime`. Explicitly set the timezone.

In [2]:
nysm_data_file = '/spare11/atm533/data/202508_nysm_merged.csv'
timeFormat = "%Y-%m-%d %H:%M:%S UTC"
nysm_data = pd.read_csv(nysm_data_file,parse_dates=['time'], date_format=timeFormat)
nysm_data['time'].dt.tz_localize(tz='UTC')

0         2025-07-31 23:00:00+00:00
1         2025-07-31 23:05:00+00:00
2         2025-07-31 23:10:00+00:00
3         2025-07-31 23:15:00+00:00
4         2025-07-31 23:20:00+00:00
                     ...           
1274359   2025-09-03 20:05:00+00:00
1274360   2025-09-03 20:10:00+00:00
1274361   2025-09-03 20:15:00+00:00
1274362   2025-09-03 20:20:00+00:00
1274363   2025-09-03 20:25:00+00:00
Name: time, Length: 1274364, dtype: datetime64[ns, UTC]

Remove unwanted columns

In [3]:
nysm_data.drop(['temp_9m [degC]',
       'avg_wind_speed_prop [m/s]', 'max_wind_speed_prop [m/s]','wind_speed_stddev_prop [m/s]', 'wind_direction_prop [degrees]',
       'wind_direction_stddev_prop [degrees]','wind_speed_stddev_sonic [m/s]','wind_direction_stddev_sonic [degrees]', 'solar_insolation [W/m^2]','snow_depth [cm]', 'frozen_soil_05cm [bit]',
       'frozen_soil_25cm [bit]', 'frozen_soil_50cm [bit]',
       'soil_temp_05cm [degC]', 'soil_temp_25cm [degC]',
       'soil_temp_50cm [degC]', 'soil_moisture_05cm [m^3/m^3]',
       'soil_moisture_25cm [m^3/m^3]', 'soil_moisture_50cm [m^3/m^3]'],inplace=True,axis='columns')

Each dataframe has varying column names. Let's standardize by creating a `dictionary` that will map current column names to common (and in some cases, much shorter) names.

In [4]:
column_mapping = {'station' : 'STID',
                  'time': 'TIME',
                  'temp_2m [degC]': 'TMPC',
                  'relative_humidity [percent]': 'RELH',
                  'precip_incremental [mm]': 'PRCP',
                  'precip_local [mm]': 'PTOT',
                  'precip_max_intensity [mm/min]': 'PRAT',
                  'avg_wind_speed_sonic [m/s]': 'SPED',
                  'max_wind_speed_sonic [m/s]': 'GUMS',
                  'wind_direction_sonic [degrees]': 'DRCT', 
                  'station_pressure [mbar]': 'PRES',
                  'stid': 'STID',
                  'name': 'NAME',
                  'lat': 'SLAT',
                  'lon': 'SLON',
                  'elevation': 'SELV',
                  'STN':  'STID',
                  'YYMMDD/HHMM': 'TIME'}

Rename the columns according to our dictionary. Then examine each Dataframe to see how they look.

In [5]:
nysm_data.rename(columns=column_mapping, inplace=True)

In [6]:
nysm_data.head()

,STID,TIME,TMPC,RELH,PRCP,PTOT,PRAT,SPED,GUMS,DRCT,PRES,SLAT,SLON,SELV,NAME
0,ADDI,2025-07-31 23:00:00,14.8,98.8,0.0,19.51,0.0,2.0,3.6,353.0,961.19,42.04036,-77.23726,507.614,Addison
1,ADDI,2025-07-31 23:05:00,14.8,99.0,0.0,19.51,0.0,2.2,3.7,351.0,961.23,42.04036,-77.23726,507.614,Addison
2,ADDI,2025-07-31 23:10:00,14.8,99.1,0.0,19.51,0.0,2.3,3.8,349.0,961.37,42.04036,-77.23726,507.614,Addison
3,ADDI,2025-07-31 23:15:00,14.8,99.2,0.0,19.51,0.0,2.7,4.2,347.0,961.45,42.04036,-77.23726,507.614,Addison
4,ADDI,2025-07-31 23:20:00,14.8,99.3,0.0,19.51,0.0,3.2,5.1,348.0,961.46,42.04036,-77.23726,507.614,Addison


The NYSM data does not contain dewpoint nor sea-level pressure yet. Calculate and create a column for dewpoint; for the purposes of this notebook, we'll not do the SLP conversion.

In [7]:
tmpc = nysm_data['TMPC'].values * units('degC')
rh = nysm_data['RELH'].values * units('percent')

In [8]:
tmpc

Magnitude,[14.8 14.8 14.8 ... 26.5 26.5 26.2]
Units,degree_Celsius


In [9]:
nysm_data['DWPC'] = mpcalc.dewpoint_from_relative_humidity(tmpc, rh)

Let's rearrange the columns into a more logical order.

In [10]:
colOrder = ['STID','TIME','TMPC','DWPC','RELH','PRES','DRCT','SPED','GUMS','PRCP','PTOT','PRAT']

In [11]:
nysm_data = nysm_data.reindex(columns=colOrder)

In [12]:
nysm_data

,STID,TIME,TMPC,DWPC,RELH,PRES,DRCT,SPED,GUMS,PRCP,PTOT,PRAT
0,ADDI,2025-07-31 23:00:00,14.8,14.604478,98.8,961.19,353.0,2.0,3.6,0.0,19.51,0.0
1,ADDI,2025-07-31 23:05:00,14.8,14.635792,99.0,961.23,351.0,2.2,3.7,0.0,19.51,0.0
2,ADDI,2025-07-31 23:10:00,14.8,14.651428,99.1,961.37,349.0,2.3,3.8,0.0,19.51,0.0
3,ADDI,2025-07-31 23:15:00,14.8,14.667051,99.2,961.45,347.0,2.7,4.2,0.0,19.51,0.0
4,ADDI,2025-07-31 23:20:00,14.8,14.682659,99.3,961.46,348.0,3.2,5.1,0.0,19.51,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1274359,YORK,2025-09-03 20:05:00,26.5,10.246598,36.1,987.94,201.0,3.4,5.8,0.0,10.40,0.0
1274360,YORK,2025-09-03 20:10:00,26.5,10.079970,35.7,987.88,195.0,3.0,5.9,0.0,10.40,0.0
1274361,YORK,2025-09-03 20:15:00,26.5,10.329303,36.3,987.86,202.0,3.6,6.6,0.0,10.40,0.0
1274362,YORK,2025-09-03 20:20:00,26.5,10.163489,35.9,987.79,217.0,2.9,5.5,0.0,10.40,0.0


### Split-Apply-Combine

Now go through the *Split-Apply-Combine* methodology, as defined by Wickham, 2011 and illustrated by McKinney, 2017 (see refs at end of notebook)

<center><img src="https://www.oreilly.com/api/v2/epubs/9781783985128/files/graphics/5128OS_09_01.jpg" alt="SplitApplyCombine" style="width: 350px;"/></center>

#### Split: Group by station id

In [13]:
grouped = nysm_data.groupby(['STID'])

In [14]:
grouped

This has *split* the DataFrame into a Pandas `groupby` object. 

#### Apply and Combine:

Now we *apply* a function on the now-split groups, which then *combines* the group and the values returned by the function into a new DataFrame. 

One built-in `groupby` function is `describe`. It returns a `DataFrame` containing summary statistics, grouped by station, for each column that contains numerical values:

In [15]:
df_stns = grouped.describe() # Depending on the size of your DataFrame, this may take a while

In [16]:
df_stns

TIME                                                      \
      count                           mean                  min   
STID                                                              
ADDI   9951  2025-08-17 23:12:01.043111168  2025-07-31 23:00:00   
ANDE   9994  2025-08-17 21:47:17.902741760  2025-07-31 23:00:00   
BATA   9955  2025-08-17 23:36:16.062280192  2025-07-31 23:00:00   
BEAC   9982  2025-08-17 19:54:18.014425856  2025-07-31 23:00:00   
BELD  10102  2025-08-18 01:52:47.164918016  2025-07-31 23:00:00   
...     ...                            ...                  ...   
WFMB   9913  2025-08-18 00:11:19.199031552  2025-07-31 23:00:00   
WGAT  10091  2025-08-17 22:00:25.359231232  2025-07-31 23:00:00   
WHIT   9917  2025-08-17 23:24:13.655339264  2025-07-31 23:00:00   
WOLC   9936  2025-08-18 00:24:05.652173824  2025-07-31 23:00:00   
YORK  10005  2025-08-18 01:10:32.953523456  2025-07-31 23:00:00   

                                                                     \
                      25%                  50%                  75%   
STID                                                                  
ADDI  2025-08-09 14:17:30  2025-08-18 03:20:00  2025-08-26 06:37:30   
ANDE  2025-08-09 12:06:15  2025-08-18 01:12:30  2025-08-26 05:33:45   
BATA  2025-08-09 14:17:30  2025-08-18 01:20:00  2025-08-26 09:57:30   
BEAC  2025-08-09 09:26:15  2025-08-17 23:37:30  2025-08-26 04:28:45   
BELD  2025-08-09 16:21:15  2025-08-18 10:02:30  2025-08-26 10:58:45   
...                   ...                  ...                  ...   
WFMB  2025-08-09 11:40:00  2025-08-18 00:55:00  2025-08-26 13:25:00   
WGAT  2025-08-09 11:12:30  2025-08-17 23:50:00  2025-08-26 07:37:30   
WHIT  2025-08-09 12:40:00  2025-08-18 01:10:00  2025-08-26 10:15:00   
WOLC  2025-08-09 13:58:45  2025-08-18 04:17:30  2025-08-26 11:46:15   
YORK  2025-08-09 14:50:00  2025-08-18 03:25:00  2025-08-26 12:00:00   

                                   TMPC             ...   PTOT            \
                      max  std    count       mean  ...    max       std   
STID                                                ...                    
ADDI  2025-09-03 20:25:00  NaN   9946.0  17.569304  ...  19.51  2.842793   
ANDE  2025-09-03 20:25:00  NaN   9981.0  16.451348  ...  28.14  3.705134   
BATA  2025-09-03 20:25:00  NaN   9950.0  19.842965  ...  20.74  3.150792   
BEAC  2025-09-03 20:25:00  NaN   9962.0  20.328117  ...  38.45  5.684207   
BELD  2025-09-03 20:25:00  NaN  10099.0  17.991039  ...  47.60  6.505500   
...                   ...  ...      ...        ...  ...    ...       ...   
WFMB  2025-09-03 20:25:00  NaN   9913.0  17.757551  ...  31.19  2.833537   
WGAT  2025-09-03 20:25:00  NaN   9974.0  16.957209  ...  11.60  2.168873   
WHIT  2025-09-03 20:25:00  NaN   9886.0  19.427989  ...  14.35  1.586485   
WOLC  2025-09-03 20:25:00  NaN   9935.0  19.663473  ...  20.52  3.772761   
YORK  2025-09-03 20:25:00  NaN  10003.0  19.050665  ...  17.52  2.896479   

         PRAT                                                
        count      mean  min  25%  50%  75%   max       std  
STID                                                         
ADDI   9942.0  0.001984  0.0  0.0  0.0  0.0  2.11  0.042700  
ANDE   9981.0  0.001878  0.0  0.0  0.0  0.0  0.88  0.023725  
BATA   9950.0  0.002518  0.0  0.0  0.0  0.0  1.28  0.039980  
BEAC   9962.0  0.004119  0.0  0.0  0.0  0.0  2.06  0.065896  
BELD  10099.0  0.003042  0.0  0.0  0.0  0.0  1.34  0.040118  
...       ...       ...  ...  ...  ...  ...   ...       ...  
WFMB   9913.0  0.002609  0.0  0.0  0.0  0.0  2.45  0.054426  
WGAT   9974.0  0.001252  0.0  0.0  0.0  0.0  1.07  0.023231  
WHIT   9886.0  0.001973  0.0  0.0  0.0  0.0  1.22  0.036289  
WOLC   9935.0  0.003218  0.0  0.0  0.0  0.0  1.06  0.035201  
YORK  10003.0  0.007977  0.0  0.0  0.0  0.0  2.86  0.082575  

[127 rows x 88 columns]

#### Analyze DataFrames produced by the S/A/C technique

<div class="alert alert-block alert-info">
    <b>Tip:</b> This is just another Pandas DataFrame, but with multiple (i.e. <i>hierarchical</i>) column indices. Specifically, each column from the original dataframe now has sub-columns, such as <code>count</code>, <code>mean</code>, and <code>50%</code>.</div>

Five-minute and daily accumulated precip:

In [17]:
df_stns[['PRCP','PTOT']]

PRCP                                                    PTOT  \
        count      mean  min  25%  50%  75%    max       std    count   
STID                                                                    
ADDI   9942.0  0.006998  0.0  0.0  0.0  0.0   5.41  0.121817   9943.0   
ANDE   9979.0  0.007654  0.0  0.0  0.0  0.0   2.88  0.074483   9980.0   
BATA   9947.0  0.008624  0.0  0.0  0.0  0.0   5.22  0.133928   9949.0   
BEAC   9958.0  0.016401  0.0  0.0  0.0  0.0   8.43  0.236654   9959.0   
BELD  10087.0  0.011262  0.0  0.0  0.0  0.0   3.38  0.114794  10089.0   
...       ...       ...  ...  ...  ...  ...    ...       ...      ...   
WFMB   9913.0  0.010188  0.0  0.0  0.0  0.0  11.29  0.216708   9913.0   
WGAT   9918.0  0.004883  0.0  0.0  0.0  0.0   2.97  0.061465   9918.0   
WHIT   9883.0  0.007424  0.0  0.0  0.0  0.0   4.52  0.121423   9885.0   
WOLC   9934.0  0.010134  0.0  0.0  0.0  0.0   3.92  0.096504   9934.0   
YORK  10001.0  0.012682  0.0  0.0  0.0  0.0   6.73  0.127458  10001.0   

                                                      
          mean  min  25%  50%   75%    max       std  
STID                                                  
ADDI  0.813360  0.0  0.0  0.0  0.00  19.51  2.842793  
ANDE  0.888693  0.0  0.0  0.0  0.00  28.14  3.705134  
BATA  0.844831  0.0  0.0  0.0  0.00  20.74  3.150792  
BEAC  1.672142  0.0  0.0  0.0  0.00  38.45  5.684207  
BELD  1.366702  0.0  0.0  0.0  0.22  47.60  6.505500  
...        ...  ...  ...  ...   ...    ...       ...  
WFMB  0.628204  0.0  0.0  0.0  0.00  31.19  2.833537  
WGAT  0.737224  0.0  0.0  0.0  0.00  11.60  2.168873  
WHIT  0.347460  0.0  0.0  0.0  0.00  14.35  1.586485  
WOLC  1.173064  0.0  0.0  0.0  0.00  20.52  3.772761  
YORK  0.944493  0.0  0.0  0.0  0.10  17.52  2.896479  

[127 rows x 16 columns]

Temperature

In [18]:
df_stns ['TMPC']

,count,mean,min,25%,50%,75%,max,std
STID,,,,,,,,
ADDI,9946.0,17.569304,2.8,13.6,17.5,21.5,31.9,5.577630
ANDE,9981.0,16.451348,2.5,12.3,16.3,20.7,31.2,5.943882
BATA,9950.0,19.842965,7.7,15.6,19.9,23.6,33.2,5.334527
BEAC,9962.0,20.328117,8.2,16.3,20.5,24.1,32.8,5.118037
BELD,10099.0,17.991039,4.6,14.0,17.9,21.6,31.5,5.104999
...,...,...,...,...,...,...,...,...
WFMB,9913.0,17.757551,7.4,13.8,17.5,21.5,30.9,4.788794
WGAT,9974.0,16.957209,2.7,12.5,16.3,21.5,31.5,5.927576
WHIT,9886.0,19.427989,6.2,14.8,19.1,23.8,34.0,5.966176


What's the maximum 5-minute precip value for the entire NYSM network for that day?

In [19]:
df_stns['PRCP']['max'].max()

np.float64(357.21)

Next we'll select just a couple of rows.

In [20]:
df_stns.loc[['VOOR','WOLC']]

TIME                                                      \
      count                           mean                  min   
STID                                                              
VOOR  10036  2025-08-17 21:45:01.285372928  2025-07-31 23:00:00   
WOLC   9936  2025-08-18 00:24:05.652173824  2025-07-31 23:00:00   

                                                                     \
                      25%                  50%                  75%   
STID                                                                  
VOOR  2025-08-09 12:38:45  2025-08-18 04:17:30  2025-08-26 04:41:15   
WOLC  2025-08-09 13:58:45  2025-08-18 04:17:30  2025-08-26 11:46:15   

                                   TMPC             ...   PTOT            \
                      max  std    count       mean  ...    max       std   
STID                                                ...                    
VOOR  2025-09-03 20:25:00  NaN  10035.0  19.273632  ...  27.67  3.815322   
WOLC  2025-09-03 20:25:00  NaN   9935.0  19.663473  ...  20.52  3.772761   

         PRAT                                                
        count      mean  min  25%  50%  75%   max       std  
STID                                                         
VOOR  10035.0  0.001838  0.0  0.0  0.0  0.0  1.69  0.041462  
WOLC   9935.0  0.003218  0.0  0.0  0.0  0.0  1.06  0.035201  

[2 rows x 88 columns]

Next, *split* by station again, *apply* the max/min functions, and *combine* into two DataFrames which contain max and min values for all measured variables:

In [21]:
df_maxes = nysm_data.groupby(['STID']).max()

In [22]:
df_mins = nysm_data.groupby(['STID']).min()

In [23]:
df_maxes

,TIME,TMPC,DWPC,RELH,PRES,DRCT,SPED,GUMS,PRCP,PTOT,PRAT
STID,,,,,,,,,,,
ADDI,2025-09-03 20:25:00,31.9,21.675981,100.0,970.37,360.0,7.4,12.6,5.41,19.51,2.11
ANDE,2025-09-03 20:25:00,31.2,21.265332,100.0,970.00,360.0,5.4,10.6,2.88,28.14,0.88
BATA,2025-09-03 20:25:00,33.2,22.282585,100.0,995.80,360.0,10.7,16.7,5.22,20.74,1.28
BEAC,2025-09-03 20:25:00,32.8,24.165299,99.9,1019.57,360.0,7.6,12.5,8.43,38.45,2.06
BELD,2025-09-03 20:25:00,31.5,22.129324,100.0,975.36,360.0,6.7,10.9,3.38,47.60,1.34
...,...,...,...,...,...,...,...,...,...,...,...
WFMB,2025-09-03 20:25:00,30.9,19.852462,99.8,959.19,360.0,3.4,11.5,11.29,31.19,2.45
WGAT,2025-09-03 20:25:00,31.5,21.244779,100.0,978.12,360.0,3.6,9.6,2.97,11.60,1.07
WHIT,2025-09-03 20:25:00,34.0,22.867300,100.0,1026.17,360.0,6.6,15.1,4.52,14.35,1.22


<div class="alert alert-block alert-warning">
<b>Tip:</b> Assign new names for these two series; else they will both be labeled TMPC!</div>

In [24]:
maxT = df_maxes['TMPC'].rename('TMax')

In [25]:
minT = df_mins['TMPC'].rename('TMin')

Create a DataFrame out of these two Series, using Pandas' `concat` function.

In [26]:
pd.concat([maxT,minT],axis='columns')

,TMax,TMin
STID,,
ADDI,31.9,2.8
ANDE,31.2,2.5
BATA,33.2,7.7
BEAC,32.8,8.2
BELD,31.5,4.6
...,...,...
WFMB,30.9,7.4
WGAT,31.5,2.7
WHIT,34.0,6.2


*Group by* date/time instead of station id, and then apply/combine.

In [27]:
grouped = nysm_data.groupby(['TIME'])

In [28]:
df_Maxes = grouped.max()
df_Mins = grouped.min()

In [29]:
df_Maxes

,STID,TMPC,DWPC,RELH,PRES,DRCT,SPED,GUMS,PRCP,PTOT,PRAT
TIME,,,,,,,,,,,
2025-07-31 23:00:00,YORK,23.2,22.262929,100.0,1016.69,360.0,5.8,9.4,1.03,78.30,0.25
2025-07-31 23:05:00,YORK,23.2,22.246085,100.0,1016.69,360.0,5.5,9.4,0.59,78.48,0.15
2025-07-31 23:10:00,YORK,23.1,22.212352,100.0,1016.70,359.0,5.9,10.2,1.40,78.76,0.36
2025-07-31 23:15:00,YORK,23.1,22.212352,99.9,1016.76,360.0,5.3,9.3,1.34,79.39,0.33
2025-07-31 23:20:00,YORK,22.9,22.328650,99.8,1016.84,360.0,5.7,10.3,1.27,79.93,0.29
...,...,...,...,...,...,...,...,...,...,...,...
2025-09-03 20:05:00,YORK,27.7,14.363374,61.0,1012.12,318.0,5.8,10.0,0.00,10.40,0.00
2025-09-03 20:10:00,YORK,27.5,14.130408,61.8,1012.14,356.0,5.9,9.1,0.00,10.40,0.00
2025-09-03 20:15:00,YORK,27.6,13.958513,64.7,1012.21,357.0,5.9,9.1,0.00,10.40,0.00


<div class="alert alert-block alert-warning">
<b>Note:</b> This is a bit deceptive since it looks like YORK had the maximum values for all parameters. But that is because the station ID, <b>STID</b> got sorted as well.</div>

We could construct a DataFrame containing just the max/mins for some selected parameters for each time.

In [30]:
maxT = df_Maxes[['TMPC','DWPC','SPED','GUMS','PRCP','PTOT','PRAT']]

In [31]:
maxT

,TMPC,DWPC,SPED,GUMS,PRCP,PTOT,PRAT
TIME,,,,,,,
2025-07-31 23:00:00,23.2,22.262929,5.8,9.4,1.03,78.30,0.25
2025-07-31 23:05:00,23.2,22.246085,5.5,9.4,0.59,78.48,0.15
2025-07-31 23:10:00,23.1,22.212352,5.9,10.2,1.40,78.76,0.36
2025-07-31 23:15:00,23.1,22.212352,5.3,9.3,1.34,79.39,0.33
2025-07-31 23:20:00,22.9,22.328650,5.7,10.3,1.27,79.93,0.29
...,...,...,...,...,...,...,...
2025-09-03 20:05:00,27.7,14.363374,5.8,10.0,0.00,10.40,0.00
2025-09-03 20:10:00,27.5,14.130408,5.9,9.1,0.00,10.40,0.00
2025-09-03 20:15:00,27.6,13.958513,5.9,9.1,0.00,10.40,0.00


<div class="alert alert-block alert-warning">
<b>Time to re-sort!:</b> If we do not re-sort the index of this DataFrame, the next cell won't work right!</div>

In [32]:
maxT = maxT.sort_index()

In [33]:
columnMap = {'TMPC': 'TMax',
             'DWPC': 'TdMax',
             'SPED': 'SPDMax',
             'GUMS': 'GUSTMax',
             'PRCP': 'P5MinMax',
             'PTOT': 'PDayMax',
             'PRAT': 'PRateMax'}
maxT.rename(columns=columnMap, inplace=True)

In [34]:
maxT

,TMax,TdMax,SPDMax,GUSTMax,P5MinMax,PDayMax,PRateMax
TIME,,,,,,,
2025-07-31 23:00:00,23.2,22.262929,5.8,9.4,1.03,78.30,0.25
2025-07-31 23:05:00,23.2,22.246085,5.5,9.4,0.59,78.48,0.15
2025-07-31 23:10:00,23.1,22.212352,5.9,10.2,1.40,78.76,0.36
2025-07-31 23:15:00,23.1,22.212352,5.3,9.3,1.34,79.39,0.33
2025-07-31 23:20:00,22.9,22.328650,5.7,10.3,1.27,79.93,0.29
...,...,...,...,...,...,...,...
2025-09-03 20:05:00,27.7,14.363374,5.8,10.0,0.00,10.40,0.00
2025-09-03 20:10:00,27.5,14.130408,5.9,9.1,0.00,10.40,0.00
2025-09-03 20:15:00,27.6,13.958513,5.9,9.1,0.00,10.40,0.00


Do the same but for minima (obviously, most of the columns will have 0 as a minimum)

In [35]:
minT = df_Mins[['TMPC','DWPC','SPED','GUMS','PRCP','PTOT','PRAT']].sort_index()
columnMap = {'TMPC': 'TMin',
             'DWPC': 'TdMin',
             'SPED': 'SPDMin',
             'GUMS': 'GUSTMin',
             'PRCP': 'P5MinMin',
             'PTOT': 'PDayMin',
             'PRAT': 'PRateMin'}
minT.rename(columns=columnMap, inplace=True)

In [36]:
minT

,TMin,TdMin,SPDMin,GUSTMin,P5MinMin,PDayMin,PRateMin
TIME,,,,,,,
2025-07-31 23:00:00,12.9,8.340882,0.1,0.5,0.0,0.0,0.0
2025-07-31 23:05:00,12.8,8.435745,0.0,0.3,0.0,0.0,0.0
2025-07-31 23:10:00,12.7,8.515581,0.0,0.3,0.0,0.0,0.0
2025-07-31 23:15:00,12.5,8.489011,0.1,0.5,0.0,0.0,0.0
2025-07-31 23:20:00,12.3,8.409047,0.1,0.5,0.0,0.0,0.0
...,...,...,...,...,...,...,...
2025-09-03 20:05:00,19.4,5.753085,0.0,0.0,0.0,0.0,0.0
2025-09-03 20:10:00,19.1,5.367325,0.0,0.0,0.0,0.0,0.0
2025-09-03 20:15:00,18.9,5.550827,0.1,0.5,0.0,0.0,0.0


<div class="alert alert-block alert-info">
<b>Note:</b> We don't see a station ID column, but that's because each max and min likely comes from a different station.</div>

---
## Summary
* Pandas' `groupby` method forms the basis of this library's *split-apply-combine* methodology.

### What's Next?
Next, we will look at tabular datasets that are so large that they are best represented in a format other than plain-text.

## Resources and References
1. [MetPy Monday 99](https://www.youtube.com/watch?v=31k53iHE-yw&list=PLQut5OXpV-0ir4IdllSt1iEZKTwFBa7kO&index=90&ab_channel=Unidata) 
1. [Split/Apply/Combine (Wickham, 2011)](http://dx.doi.org/10.18637/jss.v040.i01)
1. [Ch. 10 notebook from Github Repository for Wes McKinney's Pandas for Data Analysis book](https://github.com/wesm/pydata-book) 